In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('/home/wtim/neural-capability-maps/')

In [3]:
import torch
from tabulate import tabulate
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nrm.dataset.r3 as r3
import nrm.dataset.se3 as se3
from nrm.dataset.morphology import sample_morph
from nrm.dataset.reachability_manifold import sample_reachability_manifold, estimate_reachable_ball, \
    sample_poses_in_reach
from nrm.dataset.kinematics import analytical_inverse_kinematics, transformation_matrix, numerical_inverse_kinematics
from nrm.dataset.loader import TrainingSet

from nrm.logger import binary_confusion_matrix
from nrm.visualisation import visualise_predictions, visualise_workspace, display_geodesic, display_slice, display_sphere

from nrm.model import MLP

In [4]:
def compute_metrics(pred, label):
    (true_positives, false_negatives), (false_positives, true_negatives) = binary_confusion_matrix(pred, label)
    accuracy = (pred == label).sum() / pred.shape[0] * 100
    f1 = 2 * true_positives / (2 * true_positives + false_positives + false_negatives) * 100 if true_positives + false_positives + false_negatives != 0 else 100
    reachable = label.sum().item() / label.shape[0] * 100

    headers = ["True Positives", "True Negatives", "False Positives", "False Negatives", "F1 Score", "Accuracy",
               "Reachable"]
    table = list(
        zip([true_positives], [true_negatives], [false_positives], [false_negatives], [f1], [accuracy], [reachable]))
    print(tabulate(table, headers=headers, floatfmt=".2f"))

In [5]:
torch.manual_seed(1)

# Generate Robot & Test Cases

In [6]:
steps = 100
morph = sample_morph(1, 6, True)[0]
mat = transformation_matrix(morph[0, 0:1],
                            morph[0, 1:2],
                            morph[0, 2:3],
                            torch.zeros_like(morph[0, 0:1]))
torus_axis = torch.nn.functional.normalize(mat[:3, 2], dim=0)
centre, radius = estimate_reachable_ball(morph)
fixed_axes = torch.argmax(torus_axis.abs())
axes_mask = torch.ones(3, dtype=torch.bool)
axes_mask[fixed_axes] = False
axes_range = torch.linspace(-radius, radius, steps)

In [7]:
any_poses = se3.random(100_000)
ball_poses = se3.random_ball(100_000, *estimate_reachable_ball(morph))
poses = sample_poses_in_reach(100_000, morph)

In [130]:
any_labels = analytical_inverse_kinematics(morph.double(), any_poses.double())[1] != -1
ball_labels = analytical_inverse_kinematics(morph.double(), ball_poses.double())[1] != -1
labels = analytical_inverse_kinematics(morph.double(), poses.double())[1] != -1

In [9]:
visualise_workspace(morph, poses, labels)

In [10]:
geodesic_samples = 1000

start = poses[labels][0]
end = poses[~labels][0]

tangent = se3.log(start, end)
t = torch.linspace(0, 1, geodesic_samples).view(-1, 1)
geodesic_poses = se3.exp(start.unsqueeze(0).repeat(geodesic_samples, 1, 1), t * tangent)

In [11]:
geodesic_labels = analytical_inverse_kinematics(morph.double(), geodesic_poses.double())[1] != -1

In [12]:
anchor = poses[labels][torch.median(poses[labels][:, :3, 3].norm(dim=1), dim=0).indices]
slice_poses = anchor.unsqueeze(0).expand(steps ** 2, -1, -1).clone()
slice_poses[:, :3, 3][:, axes_mask] = centre[axes_mask]

slice_poses[:, :3, 3][:, axes_mask] += torch.stack(torch.meshgrid(axes_range, axes_range, indexing='ij'),
                                                   dim=-1).reshape(-1, 2)

In [13]:
slice_labels = analytical_inverse_kinematics(morph.double(), slice_poses.double())[1] != -1

In [14]:
sphere_anchor = poses[labels][torch.median(poses[labels][:, :3, 3].norm(dim=1), dim=0).indices]
sphere_poses = sphere_anchor.unsqueeze(0).expand(steps ** 2, -1, -1).clone()
sphere_poses[:, :3, 3] = centre

theta = torch.linspace(0, torch.pi, steps)
phi = torch.linspace(0, 2 * torch.pi, steps)
theta_grid, phi_grid = torch.meshgrid(theta, phi, indexing='ij')

x = sphere_anchor[:3, 3].norm() * torch.sin(theta_grid) * torch.cos(phi_grid)
y = sphere_anchor[:3, 3].norm() * torch.sin(theta_grid) * torch.sin(phi_grid)
z = sphere_anchor[:3, 3].norm() * torch.cos(theta_grid)

sphere_poses[:, :3, 3] = centre + torch.stack([x, y, z], dim=-1).reshape(-1, 3)

In [15]:
sphere_labels = analytical_inverse_kinematics(morph.double(), sphere_poses.double())[1] != -1

# FK-Only

In [16]:
r_indices = estimate_capability_map(morph.to("cuda"), seconds=60)

[auto-batch] est. bytes/sample: 65672 (64.13 KiB), free: 37.83 GiB, safety: 0.5, batch_size: 309256


In [17]:
nn = se3.nn(r_indices[torch.randint(0, r_indices.shape[0], (100_000,))])
nn_labels = torch.isin(nn, r_indices).sum(dim=-1)
nn_mask = nn_labels != nn.shape[-1]
nn = nn[nn_mask].unique()[:100_000]
boundary_poses = se3.cell_noisy(nn)

In [18]:
boundary_labels = analytical_inverse_kinematics(morph.double(), boundary_poses.double())[1] != -1

Any
  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           87.97             97.41               2.59              12.03       92.33       96.58         8.77


In [19]:
any_discretisation = torch.isin(se3.index(any_poses), r_indices)
ball_discretisation = torch.isin(se3.index(ball_poses), r_indices)
discretisation = torch.isin(se3.index(poses), r_indices)
boundary_discretisation = torch.isin(se3.index(boundary_poses), r_indices)
geodesic_discretisation = torch.isin(se3.index(geodesic_poses), r_indices)
slice_discretisation = torch.isin(se3.index(slice_poses), r_indices)
sphere_discretisation = torch.isin(se3.index(sphere_poses), r_indices)

Ball
  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           88.05             97.02               2.98              11.95       92.18       96.11        10.16


In [20]:
print("Boundary")
compute_metrics(boundary_discretisation, boundary_labels)

Reach
  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           87.99             91.33               8.67              12.01       89.49       90.51        24.60


In [58]:
print("Any")
compute_metrics(any_discretisation, any_labels)

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           84.18             85.66              14.34              15.82       84.81       85.40        17.70


In [21]:
print("Ball")
compute_metrics(ball_discretisation, ball_labels)

In [59]:
print("Reach")
compute_metrics(discretisation, labels)

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           79.49             96.64               3.36              20.51       86.94       92.44        24.47


In [22]:
compute_metrics(geodesic_discretisation, geodesic_labels)

In [60]:
display_geodesic([geodesic_labels, geodesic_discretisation], ["Ground-Truth", "Discretisation"])

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           86.11             95.36               4.64              13.89       90.28       92.33        32.75


In [37]:
compute_metrics(slice_discretisation, slice_labels)

In [ ]:
display_slice([slice_labels, slice_discretisation], ["Truth", "Discretisation"], morph)

In [23]:
compute_metrics(sphere_discretisation, sphere_labels)

In [27]:
display_sphere([sphere_labels, sphere_discretisation], ["Truth", "Discretisation"], sphere_anchor[:3, 3].norm())

# Numerical Inverse Kinematics

In [29]:
any_nik = numerical_inverse_kinematics(morph, any_poses)[1] != -1
ball_nik = numerical_inverse_kinematics(morph, ball_poses)[1] != -1
nik = numerical_inverse_kinematics(morph, poses)[1] != -1
boundary_nik = numerical_inverse_kinematics(morph, boundary_poses)[1] != -1
geodesic_nik = numerical_inverse_kinematics(morph, geodesic_poses)[1] != -1
slice_nik = numerical_inverse_kinematics(morph, slice_poses)[1] != -1
sphere_nik = numerical_inverse_kinematics(morph, sphere_poses)[1] != -1

Any
  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
            1.92             99.95               0.05              98.08        3.76       91.36         8.77


In [30]:
print("Boundary")
compute_metrics(boundary_nik, boundary_labels)

Ball
  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
            1.82             99.94               0.06              98.18        3.58       89.97        10.16


In [31]:
print("Any")
compute_metrics(any_nik, any_labels)

Reach
  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
            1.76             99.83               0.17              98.24        3.45       75.70        24.60


In [61]:
print("Ball")
compute_metrics(ball_nik, ball_labels)

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
            0.00             98.66               1.34             100.00        0.00       81.20        17.70


In [32]:
print("Reach")
compute_metrics(nik, labels)

In [62]:
compute_metrics(geodesic_nik, geodesic_labels)

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
            1.39            100.00               0.00              98.61        2.74       75.87        24.47


In [33]:
display_geodesic([geodesic_labels, geodesic_nik], ["Ground-Truth", "Numerical Inverse Kinematics"])

In [63]:
compute_metrics(slice_nik, slice_labels)

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
            1.25             99.87               0.13              98.75        2.47       67.57        32.75


In [34]:
display_slice([slice_labels, slice_nik], ["Truth", "Numerical Inverse Kinematics"], morph)

In [ ]:
compute_metrics(sphere_nik, sphere_labels)

In [123]:
display_sphere([sphere_labels, sphere_nik], ["Truth", "Numerical Inverse Kinematics"], sphere_anchor[:3, 3].norm())

# Training Set

In [125]:
training_set = TrainingSet(1000, False)
training_poses = []
training_labels = []

for _, pose, label in training_set:
    training_poses.append(se3.from_vector(pose))
    training_labels.append(label)

training_poses = torch.cat(training_poses, dim=0)
training_labels = torch.cat(training_labels, dim=0)

Any
  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           86.26             79.23              20.77              13.74       83.33       79.84         8.77


In [126]:
t_indices = se3.index(training_poses[training_labels]).unique()

Ball
  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           90.20             76.51              23.49               9.80       84.42       77.90        10.16


In [131]:
any_training = torch.isin(se3.index(any_poses), t_indices)
ball_training = torch.isin(se3.index(ball_poses), t_indices)
training = torch.isin(se3.index(poses), t_indices)
#boundary_training = torch.isin(se3.index(boundary_poses), t_indices)
geodesic_training = torch.isin(se3.index(geodesic_poses), t_indices)
slice_training = torch.isin(se3.index(slice_poses), t_indices)
sphere_training = torch.isin(se3.index(sphere_poses), t_indices)

Reach
  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           99.79             79.86              20.14               0.21       90.75       84.76        24.60


In [132]:
print("Boundary")
compute_metrics(boundary_training, boundary_labels)

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           56.50             56.14              43.86              43.50       56.40       56.20        17.70


In [135]:
print("Any")
compute_metrics(any_training, any_labels)

In [136]:
print("Ball")
compute_metrics(ball_training, ball_labels)

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           45.16             85.85              14.15              54.84       56.69       75.89        24.47


In [140]:
print("Reach")
compute_metrics(training, labels)

In [80]:
compute_metrics(geodesic_training, geodesic_labels)

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           37.10             82.63              17.37              62.90       48.04       67.72        32.75


In [138]:
display_geodesic([geodesic_labels, geodesic_training], ["Truth", "Training"])

In [ ]:
compute_metrics(slice_training, slice_labels)

In [43]:
display_slice([slice_labels, slice_training], ["Truth", "Training"], morph)

In [44]:
compute_metrics(sphere_training, sphere_labels)

In [115]:
display_sphere([sphere_labels, sphere_training], ["Truth", "Training"], sphere_anchor[:3, 3].norm())

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           77.89             70.33              29.67              22.11       75.05       70.99         8.77


# MLP

In [117]:
onet_coarse = MLP.from_id(1305)#OccupancyNetwork.from_id(1003)

tensor(0.5613)

In [118]:
centre, radius_s = estimate_reachable_ball(morph[:-1])  # Ignore the EEF
mat = torch.linalg.inv(transformation_matrix(morph[-1, 0:1], morph[-1, 1:2],morph[-1, 2:3],torch.zeros_like(morph[-1, 0:1])))

any_coarse = (onet_coarse.predict(morph.unsqueeze(0).expand(any_poses.shape[0], -1, -1), se3.to_vector(any_poses)) > 0) &(((any_poses @ mat)[:, :3, 3] - centre).norm(dim=-1) < radius_s)
ball_coarse = (onet_coarse.predict(morph.unsqueeze(0).expand(ball_poses.shape[0], -1, -1), se3.to_vector(ball_poses)) > 0) & (((ball_poses @ mat)[:, :3, 3] - centre).norm(dim=-1) < radius_s)
coarse = (onet_coarse.predict(morph.unsqueeze(0).expand(poses.shape[0], -1, -1), se3.to_vector(poses)) > 0) & (((poses @ mat)[:, :3, 3] - centre).norm(dim=-1) < radius_s)
coarse_boundary = (onet_coarse.predict(morph.unsqueeze(0).expand(boundary_poses.shape[0], -1, -1), se3.to_vector(boundary_poses)) > 0) & (((boundary_poses @ mat)[:, :3, 3] - centre).norm(dim=-1) < radius_s)
geodesic_coarse = (onet_coarse.predict(morph.unsqueeze(0).expand(geodesic_poses.shape[0], -1, -1), se3.to_vector(geodesic_poses)) > 0)& (((geodesic_poses @ mat)[:, :3, 3] - centre).norm(dim=-1) < radius_s)
slice_coarse = (onet_coarse.predict(morph.unsqueeze(0).expand(slice_poses.shape[0], -1, -1), se3.to_vector(slice_poses)) > 0)& (((slice_poses @ mat)[:, :3, 3] - centre).norm(dim=-1) < radius_s)
sphere_coarse = (onet_coarse.predict(morph.unsqueeze(0).expand(sphere_poses.shape[0], -1, -1), se3.to_vector(sphere_poses)) > 0)& (((sphere_poses @ mat)[:, :3, 3] - centre).norm(dim=-1) < radius_s)

In [119]:
print("Boundary")
compute_metrics(coarse_boundary, boundary_labels)

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           49.03             69.65              30.35              50.97       54.67       63.22        31.18


In [51]:
print("Any")
compute_metrics(any_coarse, any_labels)

Any
  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           77.89             82.02              17.98              22.11       79.53       81.65         8.77


In [50]:
print("Ball")
compute_metrics(ball_coarse, ball_labels)

Ball
  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           82.21             79.68              20.32              17.79       81.18       79.93        10.16


In [46]:
print("Reach")
compute_metrics(coarse, labels)

Reach
  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           99.04             90.09               9.91               0.96       94.80       92.29        24.60


In [52]:
compute_metrics(geodesic_coarse, geodesic_labels)

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           46.33             60.75              39.25              53.67       49.93       58.20        17.70


In [47]:
display_geodesic([geodesic_labels, geodesic_coarse], ["Truth", "Coarse"])

In [54]:
compute_metrics(slice_coarse, slice_labels)

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           76.13             82.83              17.17              23.87       78.77       81.19        24.47


In [112]:
display_slice([slice_labels, slice_coarse], ["Truth", "Coarse"], morph)

In [55]:
compute_metrics(sphere_coarse, sphere_labels)

  True Positives    True Negatives    False Positives    False Negatives    F1 Score    Accuracy    Reachable
----------------  ----------------  -----------------  -----------------  ----------  ----------  -----------
           72.18             59.66              40.34              27.82       67.93       63.76        32.75


In [49]:
display_sphere([sphere_labels, sphere_coarse], ["Truth", "Coarse"], sphere_anchor[:3, 3].norm())